# RAMBHA-LP — Train the PINN on Kaggle (GPU)

This notebook runs the **training** part of the pipeline on a Kaggle GPU, then hands the
trained checkpoint back to your laptop where the rest (apply + validate + figures) runs in
seconds.

**What you upload to Kaggle as a Dataset:** the `solution/` folder (or at least `src/`,
`config.yaml`) **and** the pre-generated synthetic arrays `outputs/cache/synth_train.npz` /
`synth_val.npz`. You do NOT need to upload the 800 MB RAMBHA archive — training only needs
the synthetic set, which is small. (Generate it once locally with `make synth`.)

**Steps in the Kaggle UI** (spelled out in GUIDE.md §7):
1. Create a new Notebook, Settings ▸ Accelerator ▸ **GPU T4 x1**.
2. Add Data ▸ Upload ▸ your zipped `solution` folder (it mounts at `/kaggle/input/<name>/`).
3. Run the cells below.
4. Download `pinn.pt` from the Output tab back into your local `outputs/checkpoints/`.

In [ ]:
# --- Locate the uploaded solution folder on Kaggle ---------------------------------
import os, glob, sys
cands = glob.glob('/kaggle/input/*/solution') + glob.glob('/kaggle/input/*/src/..')
SOL = None
for c in glob.glob('/kaggle/input/*/**/config.yaml', recursive=True):
    SOL = os.path.dirname(c); break
assert SOL, 'Could not find config.yaml under /kaggle/input — check your uploaded dataset.'
print('solution dir:', SOL)
sys.path.insert(0, os.path.join(SOL, 'src'))

In [ ]:
import torch, numpy as np
print('CUDA available:', torch.cuda.is_available(), '| device:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# --- Load config; bump up the training budget since we have a GPU -------------------
from rambhalp.config import load_config
cfg = load_config(os.path.join(SOL, 'config.yaml'))
cfg['train']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg['train']['epochs'] = 200          # GPU is fast; train longer than the laptop default
cfg['train']['batch_size'] = 2048
print('epochs', cfg['train']['epochs'], '| batch', cfg['train']['batch_size'])

In [ ]:
# --- Get synthetic data: prefer the uploaded cache, else regenerate here ------------
from rambhalp import synthetic
cache = None
for c in glob.glob('/kaggle/input/**/synth_train.npz', recursive=True):
    cache = os.path.dirname(c); break
if cache:
    tr = np.load(os.path.join(cache, 'synth_train.npz'))
    va = np.load(os.path.join(cache, 'synth_val.npz'))
    Xtr, Ytr, Xva, Yva = tr['X'], tr['Y'], va['X'], va['Y']
    print('loaded uploaded synthetic cache from', cache)
else:
    print('no uploaded cache — generating synthetic set on the GPU box')
    Xtr, Ytr, _ = synthetic.make_dataset(cfg['synthetic']['n_train'], cfg, seed=1337)
    Xva, Yva, _ = synthetic.make_dataset(cfg['synthetic']['n_val'], cfg, seed=1338)
print('train', Xtr.shape, '| val', Xva.shape)

In [ ]:
# --- Train the physics-informed network --------------------------------------------
from rambhalp import train
model, hist = train.train_model(Xtr, Ytr, cfg, X_val=Xva, Y_val=Yva, verbose=True)

In [ ]:
# --- Save the checkpoint to /kaggle/working (downloadable from the Output tab) ------
import json, pathlib
outdir = pathlib.Path('/kaggle/working'); outdir.mkdir(exist_ok=True)
train.save_checkpoint(model, cfg, outdir / 'pinn.pt', extra={'kind': 'pinn', 'history': hist})
json.dump(hist, open(outdir / 'train_history.json', 'w'), indent=2)
print('saved pinn.pt and train_history.json to /kaggle/working — download them into your\n'
      'local solution/outputs/checkpoints/ and solution/outputs/tables/ respectively.')

### Back on your laptop
Drop `pinn.pt` into `solution/outputs/checkpoints/` and `train_history.json` into
`solution/outputs/tables/`, then run:
```bash
python scripts/05_apply_and_recover.py
python scripts/06_validate_hop.py
python scripts/07_make_figures.py
```
That produces the recovery count, validation, and all figures from the GPU-trained model.